# IAN 603 · Final Project Starter Notebook
# AdventureWorks Sales → PostgreSQL

This notebook helps you prepare the AdventureWorks Sales Excel workbook for your final project.

It will:

1. locate the Excel workbook,
2. connect to PostgreSQL,
3. create the seven project tables,
4. read each worksheet,
5. rename Excel columns to PostgreSQL-friendly names,
6. load the workbook data into PostgreSQL,
7. validate the import,
8. leave the rest of the notebook for **your own final-project analysis**.

> **Important:** The setup cells prepare the database only. They do **not** answer the final-project analysis questions for you.

## Files you need

Keep these in the same folder when possible:

- this `.ipynb` notebook
- `AdventureWorks Sales.xlsx`

The notebook can also recognize filenames such as `AdventureWorks Sales (2).xlsx`.


## Dataset structure

The workbook contains seven worksheets that become seven PostgreSQL tables:

| Excel worksheet | PostgreSQL table |
|---|---|
| `Sales Territory_data` | `sales_territory` |
| `Reseller_data` | `reseller` |
| `Customer_data` | `customer` |
| `Product_data` | `product` |
| `Date_data` | `date_dim` |
| `Sales Order_data` | `sales_order` |
| `Sales_data` | `sales` |

The dimension/reference tables are loaded first. The `sales` fact table is loaded last because it contains foreign keys to the other tables.

### Dataset note

Some products in the workbook do not have a color value. For that reason, this notebook allows `product.color` to be `NULL` rather than inventing a value or dropping those products.


## 0. Install the required Python packages


In [ ]:
%pip install -q pandas sqlalchemy psycopg2-binary openpyxl ipywidgets


## 1. Import Python libraries and load the course database helper

The course helper is downloaded automatically if it is not already in the same folder as the notebook.


In [ ]:
from pathlib import Path
import urllib.request
import pandas as pd
from IPython.display import display

MODULE_FILE = Path("ian603_database.py")
MODULE_URL = (
    "https://raw.githubusercontent.com/"
    "ali2449/IAN-603-Preparing-Data-for-Analytics-/MSIA/"
    "ian603_database.py"
)

if not MODULE_FILE.exists():
    urllib.request.urlretrieve(MODULE_URL, MODULE_FILE)
    print("Course database helper downloaded.")

from ian603_database import IAN603Database

db = IAN603Database()
display_query = db.display_query
execute_query = db.execute_query
execute_script = db.execute_script

print("Database helper ready.")


# Part I · Find and Inspect the Excel Workbook

The next cell looks for an AdventureWorks Sales Excel file in the same folder as the notebook.

If you are using Google Colab and the file is not found, the cell will give you an opportunity to upload the workbook.


In [ ]:
def find_excel_file():
    preferred = [
        Path("AdventureWorks Sales.xlsx"),
        Path("AdventureWorks Sales (2).xlsx"),
    ]

    for candidate in preferred:
        if candidate.exists():
            return candidate

    matches = sorted(Path(".").glob("*AdventureWorks*Sales*.xlsx"))
    if matches:
        return matches[0]

    # Optional Colab fallback
    try:
        from google.colab import files
        print("AdventureWorks workbook not found. Upload the Excel file now.")
        uploaded = files.upload()
        if uploaded:
            return Path(next(iter(uploaded.keys())))
    except ImportError:
        pass

    raise FileNotFoundError(
        "AdventureWorks Sales Excel file was not found. "
        "Place the .xlsx file in the same folder as this notebook and run the cell again."
    )

EXCEL_FILE = find_excel_file()
print(f"Using workbook: {EXCEL_FILE}")


## 2. Confirm the worksheet names


In [ ]:
excel_book = pd.ExcelFile(EXCEL_FILE)
print("Worksheets found:")
for name in excel_book.sheet_names:
    print(" -", name)


## 3. Preview the workbook structure

This cell shows the number of rows and columns in each worksheet. It does not load anything into PostgreSQL yet.


In [ ]:
workbook_profile = []

for sheet_name in excel_book.sheet_names:
    preview_df = pd.read_excel(EXCEL_FILE, sheet_name=sheet_name)
    workbook_profile.append({
        "worksheet": sheet_name,
        "rows": len(preview_df),
        "columns": len(preview_df.columns),
    })

workbook_profile = pd.DataFrame(workbook_profile)
display(workbook_profile)


# Part II · Connect to PostgreSQL

Use the same PostgreSQL connection you have used in the course.


In [ ]:
db.show_connection_panel()


### Optional: Connect manually if the connection window does not work

If the PostgreSQL connection window above does not display, or the **Connect** button does not work in your Python/Jupyter environment, run the next cell instead.

You only need to use **one** connection method. The password prompt in the manual option is hidden while you type.


In [ ]:
from getpass import getpass

print("Manual PostgreSQL Connection")
print("Press Enter to use the default value shown in brackets.\n")

host = input("Host [localhost]: ").strip() or "localhost"
database = input("Database [postgres]: ").strip() or "postgres"
user = input("Username [postgres]: ").strip() or "postgres"

port_text = input("Port [5432]: ").strip()
port = int(port_text) if port_text else 5432

password = getpass("PostgreSQL password: ")

manual_config = {
    "host": host,
    "database": database,
    "user": user,
    "port": port,
    "password": password,
}

db.connect(
    config=manual_config,
    save_settings=False,
    save_password=False,
    verbose=True,
)


## 4. Confirm the PostgreSQL connection


In [ ]:
if db.connected:
    db.test_connection()
else:
    print("No active PostgreSQL connection.")
    print("Return to the connection panel above, enter your settings, and click Connect.")


# Part III · Create the Final Project Tables

The next cell **resets the project tables** and creates a clean database structure.

Run it before importing the workbook.

> If you already have work stored in tables named `sales`, `product`, `customer`, etc., do not run this cell until you have saved that work elsewhere.


In [ ]:
create_tables_sql = "DROP TABLE IF EXISTS sales CASCADE;\nDROP TABLE IF EXISTS sales_order CASCADE;\nDROP TABLE IF EXISTS product CASCADE;\nDROP TABLE IF EXISTS customer CASCADE;\nDROP TABLE IF EXISTS reseller CASCADE;\nDROP TABLE IF EXISTS date_dim CASCADE;\nDROP TABLE IF EXISTS sales_territory CASCADE;\n\nCREATE TABLE sales_territory (\n    sales_territory_key INTEGER PRIMARY KEY,\n    region              VARCHAR(100) NOT NULL,\n    country             VARCHAR(100) NOT NULL,\n    territory_group     VARCHAR(100) NOT NULL\n);\n\nCREATE TABLE reseller (\n    reseller_key    INTEGER PRIMARY KEY,\n    reseller_id     VARCHAR(20) NOT NULL,\n    business_type   VARCHAR(100) NOT NULL,\n    reseller_name   VARCHAR(200) NOT NULL,\n    city            VARCHAR(100) NOT NULL,\n    state_province  VARCHAR(100) NOT NULL,\n    country_region  VARCHAR(100) NOT NULL,\n    postal_code     VARCHAR(20) NOT NULL\n);\n\nCREATE TABLE customer (\n    customer_key    INTEGER PRIMARY KEY,\n    customer_id     VARCHAR(20) NOT NULL,\n    customer_name   VARCHAR(200) NOT NULL,\n    city            VARCHAR(100) NOT NULL,\n    state_province  VARCHAR(100) NOT NULL,\n    country_region  VARCHAR(100) NOT NULL,\n    postal_code     VARCHAR(20) NOT NULL\n);\n\nCREATE TABLE product (\n    product_key    INTEGER PRIMARY KEY,\n    sku            VARCHAR(50) NOT NULL,\n    product_name   VARCHAR(200) NOT NULL,\n    standard_cost  NUMERIC(12,4) NOT NULL,\n    color          VARCHAR(50),\n    list_price     NUMERIC(12,4) NOT NULL,\n    model          VARCHAR(150) NOT NULL,\n    subcategory    VARCHAR(100) NOT NULL,\n    category       VARCHAR(100) NOT NULL\n);\n\nCREATE TABLE date_dim (\n    date_key         INTEGER PRIMARY KEY,\n    full_date_value  DATE NOT NULL,\n    fiscal_year      VARCHAR(10) NOT NULL,\n    fiscal_quarter   VARCHAR(20) NOT NULL,\n    month_label      VARCHAR(20) NOT NULL,\n    full_date_label  VARCHAR(30) NOT NULL,\n    month_key        INTEGER NOT NULL\n);\n\nCREATE TABLE sales_order (\n    channel               VARCHAR(20) NOT NULL,\n    sales_order_line_key  BIGINT PRIMARY KEY,\n    sales_order           VARCHAR(20) NOT NULL,\n    sales_order_line      VARCHAR(30) NOT NULL\n);\n\nCREATE TABLE sales (\n    sales_order_line_key       BIGINT PRIMARY KEY,\n    reseller_key               INTEGER NOT NULL,\n    customer_key               INTEGER NOT NULL,\n    product_key                INTEGER NOT NULL,\n    order_date_key             INTEGER NOT NULL,\n    due_date_key               INTEGER NOT NULL,\n    ship_date_key              INTEGER,\n    sales_territory_key        INTEGER NOT NULL,\n    order_quantity             INTEGER NOT NULL,\n    unit_price                 NUMERIC(14,4) NOT NULL,\n    extended_amount            NUMERIC(14,4) NOT NULL,\n    unit_price_discount_pct    NUMERIC(10,6) NOT NULL,\n    product_standard_cost      NUMERIC(14,4) NOT NULL,\n    total_product_cost         NUMERIC(14,4) NOT NULL,\n    sales_amount               NUMERIC(14,4) NOT NULL,\n\n    CONSTRAINT fk_sales_order_line\n        FOREIGN KEY (sales_order_line_key)\n        REFERENCES sales_order (sales_order_line_key),\n\n    CONSTRAINT fk_sales_reseller\n        FOREIGN KEY (reseller_key)\n        REFERENCES reseller (reseller_key),\n\n    CONSTRAINT fk_sales_customer\n        FOREIGN KEY (customer_key)\n        REFERENCES customer (customer_key),\n\n    CONSTRAINT fk_sales_product\n        FOREIGN KEY (product_key)\n        REFERENCES product (product_key),\n\n    CONSTRAINT fk_sales_order_date\n        FOREIGN KEY (order_date_key)\n        REFERENCES date_dim (date_key),\n\n    CONSTRAINT fk_sales_due_date\n        FOREIGN KEY (due_date_key)\n        REFERENCES date_dim (date_key),\n\n    CONSTRAINT fk_sales_ship_date\n        FOREIGN KEY (ship_date_key)\n        REFERENCES date_dim (date_key),\n\n    CONSTRAINT fk_sales_territory\n        FOREIGN KEY (sales_territory_key)\n        REFERENCES sales_territory (sales_territory_key)\n);"

db.require_connection()
execute_script(create_tables_sql)
print("AdventureWorks project tables created.")


## 5. Confirm that the tables were created


In [ ]:
display_query("""
SELECT
    table_name
FROM information_schema.tables
WHERE table_schema = 'public'
  AND table_name IN (
      'sales_territory',
      'reseller',
      'customer',
      'product',
      'date_dim',
      'sales_order',
      'sales'
  )
ORDER BY table_name;
""", title="Final Project Tables")


# Part IV · Define the Excel → PostgreSQL Mapping

The workbook uses presentation-friendly column names. PostgreSQL uses lowercase `snake_case` names.

The mapping below is explicit so the import is predictable and easy to audit.


In [ ]:
SHEET_CONFIG = {
    "Sales Territory_data": {
        "table": "sales_territory",
        "columns": {
            "SalesTerritoryKey": "sales_territory_key",
            "Region": "region",
            "Country": "country",
            "Group": "territory_group",
        },
    },
    "Reseller_data": {
        "table": "reseller",
        "columns": {
            "ResellerKey": "reseller_key",
            "Reseller ID": "reseller_id",
            "Business Type": "business_type",
            "Reseller": "reseller_name",
            "City": "city",
            "State-Province": "state_province",
            "Country-Region": "country_region",
            "Postal Code": "postal_code",
        },
    },
    "Customer_data": {
        "table": "customer",
        "columns": {
            "CustomerKey": "customer_key",
            "Customer ID": "customer_id",
            "Customer": "customer_name",
            "City": "city",
            "State-Province": "state_province",
            "Country-Region": "country_region",
            "Postal Code": "postal_code",
        },
    },
    "Product_data": {
        "table": "product",
        "columns": {
            "ProductKey": "product_key",
            "SKU": "sku",
            "Product": "product_name",
            "Standard Cost": "standard_cost",
            "Color": "color",
            "List Price": "list_price",
            "Model": "model",
            "Subcategory": "subcategory",
            "Category": "category",
        },
    },
    "Date_data": {
        "table": "date_dim",
        "columns": {
            "DateKey": "date_key",
            "Date": "full_date_value",
            "Fiscal Year": "fiscal_year",
            "Fiscal Quarter": "fiscal_quarter",
            "Month": "month_label",
            "Full Date": "full_date_label",
            "MonthKey": "month_key",
        },
    },
    "Sales Order_data": {
        "table": "sales_order",
        "columns": {
            "Channel": "channel",
            "SalesOrderLineKey": "sales_order_line_key",
            "Sales Order": "sales_order",
            "Sales Order Line": "sales_order_line",
        },
    },
    "Sales_data": {
        "table": "sales",
        "columns": {
            "SalesOrderLineKey": "sales_order_line_key",
            "ResellerKey": "reseller_key",
            "CustomerKey": "customer_key",
            "ProductKey": "product_key",
            "OrderDateKey": "order_date_key",
            "DueDateKey": "due_date_key",
            "ShipDateKey": "ship_date_key",
            "SalesTerritoryKey": "sales_territory_key",
            "Order Quantity": "order_quantity",
            "Unit Price": "unit_price",
            "Extended Amount": "extended_amount",
            "Unit Price Discount Pct": "unit_price_discount_pct",
            "Product Standard Cost": "product_standard_cost",
            "Total Product Cost": "total_product_cost",
            "Sales Amount": "sales_amount",
        },
    },
}

LOAD_ORDER = [
    "Sales Territory_data",
    "Reseller_data",
    "Customer_data",
    "Product_data",
    "Date_data",
    "Sales Order_data",
    "Sales_data",
]

pd.DataFrame([
    {"worksheet": sheet, "table": SHEET_CONFIG[sheet]["table"]}
    for sheet in LOAD_ORDER
])


# Part V · Prepare Each Worksheet for Import

This helper function:

- reads one worksheet,
- renames its columns,
- keeps postal codes as text,
- converts the date dimension to a true date,
- preserves missing `ship_date_key` values,
- keeps missing product colors as `NULL`.


In [ ]:
def prepare_sheet(sheet_name):
    config = SHEET_CONFIG[sheet_name]

    dtype_options = None
    if sheet_name in {"Reseller_data", "Customer_data"}:
        dtype_options = {"Postal Code": str}

    df = pd.read_excel(
        EXCEL_FILE,
        sheet_name=sheet_name,
        dtype=dtype_options,
    )

    expected_excel_columns = list(config["columns"].keys())
    missing_columns = [c for c in expected_excel_columns if c not in df.columns]

    if missing_columns:
        raise ValueError(
            f"{sheet_name} is missing expected columns: {missing_columns}"
        )

    df = df[expected_excel_columns].rename(columns=config["columns"]).copy()

    if sheet_name == "Date_data":
        df["full_date_value"] = pd.to_datetime(
            df["full_date_value"],
            errors="raise"
        ).dt.date

    if sheet_name == "Sales_data":
        # ShipDateKey is allowed to be missing.
        df["ship_date_key"] = pd.to_numeric(
            df["ship_date_key"],
            errors="coerce"
        ).astype("Int64")

    return df

# Preview one prepared sheet before importing anything.
prepared_product = prepare_sheet("Product_data")
display(prepared_product.head())
print("Rows:", len(prepared_product))
print("Missing product colors:", prepared_product["color"].isna().sum())


# Part VI · Load the Excel Workbook into PostgreSQL

The import follows the foreign-key-safe order shown above.

The cell first empties the project tables so it can be rerun without creating duplicate primary keys.

Large worksheets are inserted in chunks.


In [ ]:
db.require_connection()

# Make the import cell safe to rerun.
execute_script("""
TRUNCATE TABLE
    sales,
    sales_order,
    product,
    customer,
    reseller,
    date_dim,
    sales_territory
CASCADE;
""")

source_counts = {}

for sheet_name in LOAD_ORDER:
    config = SHEET_CONFIG[sheet_name]
    table_name = config["table"]

    print(f"Loading {sheet_name} -> {table_name} ...")

    df = prepare_sheet(sheet_name)
    source_counts[table_name] = len(df)

    df.to_sql(
        table_name,
        db.engine,
        if_exists="append",
        index=False,
        chunksize=1000,
        method="multi",
    )

    print(f"  loaded {len(df):,} rows")

print("\nWorkbook import complete.")


# Part VII · Validate the Import

A successful import should have the same row count in Excel and PostgreSQL for every table.


In [ ]:
validation_rows = []

for table_name, excel_rows in source_counts.items():
    result = execute_query(
        f"SELECT COUNT(*) AS row_count FROM {table_name};"
    )
    postgres_rows = int(result.loc[0, "row_count"])

    validation_rows.append({
        "table": table_name,
        "excel_rows": excel_rows,
        "postgres_rows": postgres_rows,
        "match": excel_rows == postgres_rows,
    })

validation_df = pd.DataFrame(validation_rows)
display(validation_df)

if validation_df["match"].all():
    print("✅ All Excel and PostgreSQL row counts match.")
else:
    print("⚠ Review any table where match = False.")


## 6. Confirm key relationships


In [ ]:
display_query("""
SELECT
    COUNT(*) AS sales_rows,
    COUNT(so.sales_order_line_key) AS matched_sales_orders,
    COUNT(r.reseller_key) AS matched_resellers,
    COUNT(c.customer_key) AS matched_customers,
    COUNT(p.product_key) AS matched_products,
    COUNT(d.date_key) AS matched_order_dates,
    COUNT(t.sales_territory_key) AS matched_territories
FROM sales AS s
LEFT JOIN sales_order AS so
    ON so.sales_order_line_key = s.sales_order_line_key
LEFT JOIN reseller AS r
    ON r.reseller_key = s.reseller_key
LEFT JOIN customer AS c
    ON c.customer_key = s.customer_key
LEFT JOIN product AS p
    ON p.product_key = s.product_key
LEFT JOIN date_dim AS d
    ON d.date_key = s.order_date_key
LEFT JOIN sales_territory AS t
    ON t.sales_territory_key = s.sales_territory_key;
""", title="Relationship Validation")


## 7. Preview the loaded database


In [ ]:
display_query("""
SELECT
    s.sales_order_line_key,
    so.channel,
    p.product_name,
    p.category,
    t.region,
    s.order_quantity,
    s.sales_amount
FROM sales AS s
JOIN sales_order AS so
    ON so.sales_order_line_key = s.sales_order_line_key
JOIN product AS p
    ON p.product_key = s.product_key
JOIN sales_territory AS t
    ON t.sales_territory_key = s.sales_territory_key
ORDER BY s.sales_order_line_key
LIMIT 10;
""", title="AdventureWorks Database Preview")


# Part VIII · Your Final Project Work Begins Here

The database is now ready for your analysis.

Use the Final Project instructions in Canvas to decide what questions, cleaning steps, transformations, joins, summaries, and visual evidence you need.

## Recommended notebook style

For each project question:

1. add a short Markdown explanation of what you are trying to answer,
2. write your SQL,
3. run the query,
4. leave the result visible,
5. add a short interpretation of what you found.

Do not add SQL just to make the notebook longer. Use the course techniques when they help answer your project questions.


## Project Analysis 1

Describe the question or task you are addressing in this section.


In [ ]:
project_query_1 = """
-- Write your SQL here.

"""

# When your query is ready, remove the # from the next line.
# display_query(project_query_1, title="Project Analysis 1")


### Interpretation

Write a short explanation of what the result means and why it matters for your project.


## Project Analysis 2

Describe the question or task you are addressing in this section.


In [ ]:
project_query_2 = """
-- Write your SQL here.

"""

# When your query is ready, remove the # from the next line.
# display_query(project_query_2, title="Project Analysis 2")


### Interpretation

Write a short explanation of what the result means and why it matters for your project.


## Project Analysis 3

Describe the question or task you are addressing in this section.


In [ ]:
project_query_3 = """
-- Write your SQL here.

"""

# When your query is ready, remove the # from the next line.
# display_query(project_query_3, title="Project Analysis 3")


### Interpretation

Write a short explanation of what the result means and why it matters for your project.


## Project Analysis 4

Describe the question or task you are addressing in this section.


In [ ]:
project_query_4 = """
-- Write your SQL here.

"""

# When your query is ready, remove the # from the next line.
# display_query(project_query_4, title="Project Analysis 4")


### Interpretation

Write a short explanation of what the result means and why it matters for your project.


# Final Review Before Submission

Before submitting your project notebook, check that:

- [ ] the workbook loaded successfully,
- [ ] Excel and PostgreSQL row counts match,
- [ ] your SQL cells run without errors,
- [ ] important query results are visible,
- [ ] your explanations are written in your own words,
- [ ] your project answers the requirements in Canvas,
- [ ] your notebook is saved after the final run.

## Submission format

A Jupyter notebook (`.ipynb`) is a good option because it can keep your SQL, explanations, and results together.

If your Final Project instructions allow a `.sql` file, that format is also acceptable.
